In [1]:
import ipywidgets as widgets
from IPython.display import display

In [ ]:
import serial
import serial.tools.list_ports
import json
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import os

# -------------------------
# Cargar datos existentes
# -------------------------
datos = []

if os.path.exists("datos_celda.json"):
    with open("datos_celda.json") as f:
        for i, linea in enumerate(f):
            linea = linea.strip()
            
            if not linea:
                continue
            
            try:
                datos.append(json.loads(linea))
            except Exception as e:
                print(f"Error en línea {i}: {linea}")
                print(e)

# Crear DataFrame correctamente
df = pd.DataFrame(datos) if datos else pd.DataFrame(columns=["Peso", "lectura", "Hora"])

# -------------------------
# Función para listar puertos
# -------------------------
def listar_puertos():
    puertos = serial.tools.list_ports.comports()
    return [p.device for p in puertos]

# -------------------------
# Widgets
# -------------------------

puertos_dropdown = widgets.Text(
    value='/dev/pts/3',   # puedes cambiarlo
    description='Puerto:'
)

peso_input = widgets.FloatText(
    description='Peso (g):'
)

boton_conectar = widgets.Button(description="Conectar")
boton_guardar = widgets.Button(description="Guardar dato")

salida = widgets.Output()

display(puertos_dropdown, peso_input, boton_conectar, boton_guardar, salida)

# -------------------------
# Variables globales
# -------------------------
ser = None

# -------------------------
# Conectar serial
# -------------------------
def conectar(b):
    global ser
    with salida:
        salida.clear_output()
        try:
            ser = serial.Serial(puertos_dropdown.value, 115200, timeout=1)
            print(f"Conectado a {puertos_dropdown.value}")
        except Exception as e:
            print("Error:", e)

boton_conectar.on_click(conectar)

# -------------------------
# Guardar dato
# -------------------------
def guardar(b):
    global ser, df
    
    with salida:
        salida.clear_output()
        
        if ser is None:
            print("Primero conecta el puerto")
            return
        
        try:
            # -------------------------
            # Leer último dato del buffer
            # -------------------------
            linea = ""
            while ser.in_waiting:
                linea = ser.readline().decode(errors='ignore').strip()
            
            if linea == "":
                print("No se recibió dato")
                return
            
            print("RAW:", linea)  # debug
            
            lectura = float(linea)
            peso = peso_input.value
            
            dato = {
                "Peso": peso,
                "lectura": lectura,
                "Hora": datetime.now().isoformat()
            }
            
            # -------------------------
            # Guardar en archivo (NDJSON)
            # -------------------------
            with open("datos_celda.json", "a") as f:
                json.dump(dato, f)
                f.write("\n")
            
            # -------------------------
            # Agregar a DataFrame
            # -------------------------
            df = pd.concat([df, pd.DataFrame([dato])], ignore_index=True)
            
            print("Guardado:", dato)
            display(df.tail())
        
        except Exception as e:
            print("Error:", e)

boton_guardar.on_click(guardar)

Text(value='/dev/pts/3', description='Puerto:')

FloatText(value=0.0, description='Peso (g):')

Button(description='Conectar', style=ButtonStyle())

Button(description='Guardar dato', style=ButtonStyle())

Output()

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 63 entries, 0 to 62
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Peso     63 non-null     float64
 1   lectura  63 non-null     float64
 2   Hora     63 non-null     str    
dtypes: float64(2), str(1)
memory usage: 1.6 KB


In [ ]:
!pip install matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 644.9 kB/s eta 0:00:001m995.4 kB/s eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.7/118.7 kB 353.2 kB/s eta 0:00:001m354.6 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 41.3 kB/s eta 0:00:00m eta 0:00:010:00:07m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.6/362.6 kB 133.2 kB/s eta 0:00:00m eta 0:00:010:01:01
   ━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/5.0 MB 101.2 kB/s eta 0:00:31:29

In [8]:
import numpy as np
import matplotlib.pyplot as plt

if df.empty:
    print("No hay datos")
else:
    # -------------------------
    # AGRUPACIÓN POR PESO
    # -------------------------
    df_group = df.groupby("Peso").agg(
        lectura_mean=("lectura", "mean"),
        lectura_var=("lectura", "var"),
        count=("lectura", "count")
    ).reset_index()

    print("=== Datos agrupados ===")
    display(df_group)

    # -------------------------
    # REGRESIÓN LINEAL
    # -------------------------
    x = df_group["Peso"].values
    y = df_group["lectura_mean"].values

    coef = np.polyfit(x, y, 1)  # grado 1
    pendiente, intercepto = coef

    print(f"\nRegresión: lectura = {pendiente:.6f} * Peso + {intercepto:.6f}")

    # -------------------------
    # VARIANZA GLOBAL
    # -------------------------
    var_total = df["lectura"].var()
    print(f"Varianza total: {var_total:.6f}")

    # -------------------------
    # RECTA PARA GRAFICAR
    # -------------------------
    x_fit = np.linspace(min(x), max(x), 100)
    y_fit = pendiente * x_fit + intercepto

    # -------------------------
    # GRÁFICA
    # -------------------------
    plt.figure()

    # Datos crudos
    plt.scatter(df["Peso"], df["lectura"], label="Datos crudos", alpha=0.5)

    # Promedios por peso
    plt.scatter(x, y, marker='x', label="Promedio por peso")

    # Recta de regresión
    plt.plot(x_fit, y_fit, label="Regresión lineal")

    plt.xlabel("Peso")
    plt.ylabel("Lectura")
    plt.title("Calibración celda de carga")
    plt.legend()
    plt.grid()

    plt.show()

ModuleNotFoundError: No module named 'matplotlib'